# KV-Cache Compression Benchmarks (SnapKV, H2O, Sliding Window) on NVIDIA Tesla T4

Turnkey 1-click benchmark notebook for `Qwen/Qwen2.5-3B-Instruct` (FP16 native Tensor Cores, $O(L)$ memory-safe attention scoring, strict RoPE position preservation).

**Runtime:** ~6–10 minutes on a standard Google Colab T4 (16 GB VRAM).

In [ ]:
# Cell 1: Hardware Check, Repository Setup & Dependencies
!nvidia-smi

import os
if not os.path.isdir("src"):
    if os.path.isdir("../src"):
        %cd ..
    else:
        if not os.path.isdir("kv-cache-compression"):
            !git clone https://github.com/Medmad710/kv-cache-compression.git
        %cd kv-cache-compression

!pwd
!pip install -q -r requirements.txt

In [ ]:
# Cell 2: Fast Unit Test Suite (Tiny In-Memory Qwen2Config)
!PYTHONPATH=. pytest tests/ -v

In [ ]:
# Cell 3: Run Full T4 Benchmark Suite (Exp 1: NIAH + Exp 2: Budget Ablation)
!PYTHONPATH=. python benchmarks/run_benchmark.py --output benchmarks/results.json

In [ ]:
# Cell 4: Generate Publication Plots, Print Markdown Key Results Table & Auto-Download Artifacts
!PYTHONPATH=. python benchmarks/plot_results.py --input benchmarks/results.json --assets-dir assets

import json
import zipfile
import torch
from IPython.display import Image, display

display(Image("assets/niah_heatmaps.png"))
display(Image("assets/budget_ablation.png"))

with open("benchmarks/results.json", "r", encoding="utf-8") as f:
    res = json.load(f)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else res["metadata"].get("gpu_name", "CPU")
dtype_str = res["metadata"]["dtype"]
model_id = res["metadata"]["model_id"]
budget_pct = int(round(res["metadata"]["exp1_budget_ratio"] * 100))
summary = res["summary"]

full_kv = summary["full"]["kv_cache_mb_at_max_ctx"]
print(f"\n### Key Results ({model_id} | {gpu_name} | {dtype_str})\n")
print("| Policy | KV Budget | NIAH Accuracy (%) | KV-Cache @ 8K (MiB) | KV RAM Saved (%) | Peak VRAM (MiB) | Avg Decode (tok/s) |")
print("| :--- | :---: | :---: | :---: | :---: | :---: | :---: |")
labels = {
    "full": ("Full KV-Cache", "100%"),
    "sliding_window": ("Sliding Window + Sinks", f"{budget_pct}%"),
    "h2o": ("H2O (Heavy-Hitter Oracle)", f"{budget_pct}%"),
    "snapkv": ("SnapKV (Observation Pooled)", f"{budget_pct}%"),
}
for key in ["full", "sliding_window", "h2o", "snapkv"]:
    s = summary[key]
    name, bstr = labels[key]
    saved = (1.0 - s["kv_cache_mb_at_max_ctx"] / full_kv) * 100.0 if full_kv > 0 else 0.0
    print(
        f"| **{name}** | {bstr} | {s['niah_overall_acc_pct']:.1f}% | "
        f"{s['kv_cache_mb_at_max_ctx']:.2f} | {saved:.1f}% | "
        f"{s['peak_vram_mb']:.1f} | {s['avg_decode_tps']:.1f} |"
    )

zip_path = "t4_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in ["assets/niah_heatmaps.png", "assets/budget_ablation.png", "benchmarks/results.json"]:
        zf.write(fn)
print(f"\nPackaged artifacts into {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Google Colab; download t4_artifacts.zip manually.")